# Trabalho 2 — Classificação de NCM a partir de XPROD com CNN

Leonardo Kiepper

Dataset: ELEVEN / DESCMERC (https://github.com/vinidiol/descmerc). Entrada: `XPROD`; alvo: `NCM`.

In [2]:
%pip install tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.9/572.9 MB 11.5 MB/s  0:00:50m0:00:0100:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 10.3 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 11.6 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 11.3 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 11.5 MB/s  0:00:02 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20/20 [tensorflow]0 [tensorflow]
Note: you may need to restart the kernel to use updated packages.


In [3]:
#@title Importações
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

I0000 00:00:1794348306.885061    3181 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794348306.885686    3181 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1794348306.920029    3181 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794348307.998281    3181 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.

In [4]:
#@title Carrega a base
LOCAL = 'descmerc/'
REMOTO = 'https://github.com/vinidiol/descmerc/raw/main/'
ROOT = LOCAL if os.path.exists(LOCAL + 'BaseDESC_NCM.zip') else REMOTO

base = pd.read_csv(ROOT + 'BaseDESC_NCM.zip', sep='\t', dtype={'NCM': str})
X = base['XPROD'].astype(str).values
y = LabelEncoder().fit_transform(base['NCM'])
n_classes = int(y.max()) + 1
print(base.shape, n_classes)

(1117623, 4) 2174


In [5]:
#@title Divisão treino/teste
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=0)

In [6]:
#@title Vetorização do texto (tokens -> inteiros)
MAX_TOKENS = 50000
SEQ_LEN = 32

vetorizador = keras.layers.TextVectorization(max_tokens=MAX_TOKENS, output_sequence_length=SEQ_LEN)
vetorizador.adapt(X_tr)
S_tr = vetorizador(X_tr)
S_te = vetorizador(X_te)

E0000 00:00:1794348313.398385    3181 cuda_executor.cc:1737] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1794348313.398649    3276 cuda_executor.cc:1755] Failed to determine cuDNN version (Note that this is expected if the application doesn't link the cuDNN plugin): INTERNAL: cuDNN error: CUDNN_STATUS_INTERNAL_ERROR
W0000 00:00:1794348313.420127    3181 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [7]:
#@title Modelo: Embedding -> Conv1D -> GlobalMaxPooling -> Dense
modelo = keras.Sequential([
    keras.Input(shape=(SEQ_LEN,)),
    keras.layers.Embedding(MAX_TOKENS, 128),
    keras.layers.Conv1D(128, 3, activation='relu'),
    keras.layers.GlobalMaxPooling1D(),
    keras.layers.Dense(n_classes, activation='softmax'),
])
modelo.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
modelo.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 32, 128)        │     6,400,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 30, 128)        │        49,280 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ (None, 128)            │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 2174)           │       280,446 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,729,726 (25.67 MB)

 Trainable params: 6,729,726 (25.67 MB)

 Non-trainable params: 0 (0.00 B)

In [8]:
#@title Treinamento
hist = modelo.fit(S_tr, y_tr, epochs=5, batch_size=512, validation_split=0.1)

Epoch 1/5
1572/1572 ━━━━━━━━━━━━━━━━━━━━ 81s 51ms/step - accuracy: 0.6334 - loss: 1.8918 - val_accuracy: 0.7847 - val_loss: 0.9536
Epoch 2/5
1572/1572 ━━━━━━━━━━━━━━━━━━━━ 87s 56ms/step - accuracy: 0.8221 - loss: 0.7488 - val_accuracy: 0.8289 - val_loss: 0.7256
Epoch 3/5
1572/1572 ━━━━━━━━━━━━━━━━━━━━ 84s 53ms/step - accuracy: 0.8624 - loss: 0.5431 - val_accuracy: 0.8440 - val_loss: 0.6488
Epoch 4/5
1572/1572 ━━━━━━━━━━━━━━━━━━━━ 82s 52ms/step - accuracy: 0.8860 - loss: 0.4278 - val_accuracy: 0.8518 - val_loss: 0.6217
Epoch 5/5
1572/1572 ━━━━━━━━━━━━━━━━━━━━ 83s 53ms/step - accuracy: 0.9041 - loss: 0.3492 - val_accuracy: 0.8559 - val_loss: 0.6127


In [9]:
#@title Teste
perda, acuracia = modelo.evaluate(S_te, y_te)
print(f'Acurácia no teste: {acuracia:.4f}')

6986/6986 ━━━━━━━━━━━━━━━━━━━━ 12s 2ms/step - accuracy: 0.8515 - loss: 0.6346
Acurácia no teste: 0.8515
